In [2]:
# ============================================================
# Silver Layer: Flatten nested JSON, clean, standardize
# Source: Bronze_Lakehouse.bronze_countries_raw
# Target: Silver_Lakehouse — clean Delta tables
# ============================================================

from pyspark.sql.functions import (
    col, when, trim, upper, lower, lit, coalesce,
    explode_outer, map_keys, map_values,
    size, array_join, current_timestamp,
    regexp_replace, round as spark_round
)
from pyspark.sql.types import DoubleType, LongType, IntegerType

df_bronze = spark.table("Bronze_Lakehouse.dbo.bronze_countries_raw")

print(f"Bronze rows: {df_bronze.count()}")
df_bronze.printSchema()

StatementMeta(, 3f746ad1-7a07-4b94-af06-ccf436ba0d77, 4, Finished, Available, Finished, False)

Bronze rows: 12
root
 |-- data: struct (nullable = true)
 |    |-- meta: struct (nullable = true)
 |    |    |-- count: long (nullable = true)
 |    |    |-- duration: long (nullable = true)
 |    |    |-- limit: long (nullable = true)
 |    |    |-- more: boolean (nullable = true)
 |    |    |-- offset: long (nullable = true)
 |    |    |-- request_id: string (nullable = true)
 |    |    |-- total: long (nullable = true)
 |    |-- objects: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- _meta: struct (nullable = true)
 |    |    |    |    |-- lastUpdatedTimestamp: long (nullable = true)
 |    |    |    |-- area: struct (nullable = true)
 |    |    |    |    |-- kilometers: double (nullable = true)
 |    |    |    |    |-- miles: double (nullable = true)
 |    |    |    |-- assets: array (nullable = true)
 |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |-- borders: array (nullable = true)
 |    |    |    | 

In [6]:
# ============================================================
# Silver Layer: Flatten nested JSON, clean, standardize
# Source: bronze_countries_raw
# Target: silver_countries — clean Delta table
# ============================================================

from pyspark.sql.functions import (
    col, when, coalesce, lit, size, array_join, 
    current_timestamp, explode_outer
)
from pyspark.sql.types import DoubleType, LongType

# Read from Bronze table
df_bronze = spark.table("Bronze_Lakehouse.dbo.bronze_countries_raw")

print(f"Bronze rows: {df_bronze.count()}")

# Explode the nested array and map to correct schema paths
df_silver_countries = df_bronze.select(
    explode_outer("data.objects").alias("c")
).select(
    # --- Identity fields ---
    col("c.codes.alpha_2").alias("country_code_2"),
    col("c.codes.alpha_3").alias("country_code_3"),
    col("c.codes.ccn3").alias("country_code_numeric"),
    col("c.codes.cioc").alias("olympic_code"),

    # --- Name fields (nested struct) ---
    col("c.names.common").alias("country_name"),
    col("c.names.official").alias("official_name"),

    # --- Capital (array of structs — extract name of first capital) ---
    when(size(col("c.capitals")) > 0, col("c.capitals.name")[0])
        .otherwise(lit("N/A")).alias("capital"),

    # --- Geography ---
    coalesce(col("c.region"), lit("N/A")).alias("region"),
    coalesce(col("c.subregion"), lit("N/A")).alias("subregion"),
    col("c.continents")[0].alias("continent"),
    col("c.coordinates.lat").alias("latitude"),
    col("c.coordinates.lng").alias("longitude"),
    col("c.area.kilometers").cast(DoubleType()).alias("area_km2"),
    col("c.landlocked").alias("is_landlocked"),

    # --- Population ---
    col("c.population").cast(LongType()).alias("population"),

    # --- Borders (array → comma-separated string) ---
    when(col("c.borders").isNull(), lit("None"))
        .otherwise(array_join(col("c.borders"), ", ")).alias("bordering_countries"),
    when(col("c.borders").isNull(), 0)
        .otherwise(size(col("c.borders"))).alias("border_count"),

    # --- Currency (extract code from currency array struct) ---
    when(size(col("c.currencies")) > 0, col("c.currencies.code")[0])
        .otherwise(lit("N/A")).alias("primary_currency_code"),

    # --- Language count ---
    when(col("c.languages").isNull(), 0)
        .otherwise(size(col("c.languages"))).alias("language_count"),

    # --- Flag ---
    col("c.flag.emoji").alias("flag_emoji"),

    # --- UN membership ---
    coalesce(col("c.memberships.un"), lit(False)).alias("is_un_member"),

    # --- Timezone (first timezone) ---
    when(size(col("c.timezones")) > 0, col("c.timezones")[0])
        .otherwise(lit("N/A")).alias("primary_timezone"),

    # --- Google Maps ---
    col("c.links.google_maps").alias("google_maps_url"),

    # --- Metadata ---
    current_timestamp().alias("dwh_created_date")
).filter(
    col("country_name").isNotNull()
)

# Write to Silver Delta table
df_silver_countries.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_countries")

print(f"silver_countries rows: {df_silver_countries.count()}")
df_silver_countries.show(5, truncate=50)

StatementMeta(, 3f746ad1-7a07-4b94-af06-ccf436ba0d77, 8, Finished, Available, Finished, False)

Bronze rows: 12
silver_countries rows: 254
+--------------+--------------+--------------------+------------+-------------------+----------------------------------+-------------+--------+------------------+-------------+-----------+------------+--------+-------------+----------+-------------------+------------+---------------------+--------------+----------+------------+----------------+-------------------------------------+--------------------------+
|country_code_2|country_code_3|country_code_numeric|olympic_code|       country_name|                     official_name|      capital|  region|         subregion|    continent|   latitude|   longitude|area_km2|is_landlocked|population|bordering_countries|border_count|primary_currency_code|language_count|flag_emoji|is_un_member|primary_timezone|                      google_maps_url|          dwh_created_date|
+--------------+--------------+--------------------+------------+-------------------+----------------------------------+-------------

In [7]:
# ============================================================
# silver_country_currencies: One row per country-currency pair
# Explode nested data.objects, then explode the currencies array
# ============================================================

from pyspark.sql.functions import explode_outer, col, lit, current_timestamp, coalesce

# 1. Explode the outer objects array, then explode the currencies array
df_currencies = df_bronze.select(
    explode_outer("data.objects").alias("c")
).select(
    col("c.codes.alpha_3").alias("country_code_3"),
    col("c.names.common").alias("country_name"),
    explode_outer("c.currencies").alias("curr")
).select(
    col("country_code_3"),
    col("country_name"),
    coalesce(col("curr.code"), lit("N/A")).alias("currency_code"),
    coalesce(col("curr.name"), lit("N/A")).alias("currency_name"),
    coalesce(col("curr.symbol"), lit("N/A")).alias("currency_symbol"),
    current_timestamp().alias("dwh_created_date")
)

# Write to Silver Delta table
df_currencies.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_country_currencies")

print(f"silver_country_currencies rows: {df_currencies.count()}")
df_currencies.show(10, truncate=40)

StatementMeta(, 3f746ad1-7a07-4b94-af06-ccf436ba0d77, 9, Finished, Available, Finished, False)

silver_country_currencies rows: 276
+--------------+------------------------+-------------+--------------------------+---------------+-------------------------+
|country_code_3|            country_name|currency_code|             currency_name|currency_symbol|         dwh_created_date|
+--------------+------------------------+-------------+--------------------------+---------------+-------------------------+
|           TLS|             Timor-Leste|          USD|      United States dollar|              $|2026-10-25 11:04:35.48425|
|           TGO|                    Togo|          XOF|    West African CFA franc|             Fr|2026-10-25 11:04:35.48425|
|           TKL|                 Tokelau|          NZD|        New Zealand dollar|              $|2026-10-25 11:04:35.48425|
|           TON|                   Tonga|          TOP|            Tongan paʻanga|             T$|2026-10-25 11:04:35.48425|
|           TTO|     Trinidad and Tobago|          TTD|Trinidad and Tobago dollar|       

In [8]:
# ============================================================
# silver_country_languages: One row per country-language pair
# ============================================================

from pyspark.sql.functions import explode_outer, col, lit, current_timestamp, coalesce

# 1. Explode the outer objects array, then explode the languages array
df_languages = df_bronze.select(
    explode_outer("data.objects").alias("c")
).select(
    col("c.codes.alpha_3").alias("country_code_3"),
    col("c.names.common").alias("country_name"),
    coalesce(col("c.region"), lit("N/A")).alias("region"),
    explode_outer("c.languages").alias("lang")
).select(
    col("country_code_3"),
    col("country_name"),
    col("region"),
    coalesce(col("lang.iso639_1"), col("lang.iso639_3"), lit("N/A")).alias("language_code"),
    coalesce(col("lang.name"), lit("N/A")).alias("language_name"),
    current_timestamp().alias("dwh_created_date")
)

# Write to Silver Delta table
df_languages.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_country_languages")

print(f"silver_country_languages rows: {df_languages.count()}")
df_languages.show(10, truncate=40)

StatementMeta(, 3f746ad1-7a07-4b94-af06-ccf436ba0d77, 10, Finished, Available, Finished, False)

silver_country_languages rows: 427
+--------------+-------------------+--------+-------------+-------------+--------------------------+
|country_code_3|       country_name|  region|language_code|language_name|          dwh_created_date|
+--------------+-------------------+--------+-------------+-------------+--------------------------+
|           TLS|        Timor-Leste|    Asia|           pt|   Portuguese|2026-10-25 11:06:10.637009|
|           TLS|        Timor-Leste|    Asia|             |        Tetum|2026-10-25 11:06:10.637009|
|           TGO|               Togo|  Africa|           fr|       French|2026-10-25 11:06:10.637009|
|           TKL|            Tokelau| Oceania|           en|      English|2026-10-25 11:06:10.637009|
|           TKL|            Tokelau| Oceania|           sm|       Samoan|2026-10-25 11:06:10.637009|
|           TKL|            Tokelau| Oceania|             |    Tokelauan|2026-10-25 11:06:10.637009|
|           TON|              Tonga| Oceania|           

In [9]:
# ============================================================
# Silver Layer Validation Script
# ============================================================

from pyspark.sql.functions import col

print("=== SILVER LAYER VALIDATION ===\n")

tables = {
    "silver_countries": ["country_code_2", "region", "population"],
    "silver_country_currencies": ["country_code_3", "currency_code"],
    "silver_country_languages": ["country_code_3", "language_name"]
}

for table, key_cols in tables.items():
    df = spark.table(table)
    print(f"--- {table} ---")
    print(f"  Rows: {df.count()}")
    for c in key_cols:
        nulls = df.filter(col(c).isNull()).count()
        print(f"  Nulls in {c}: {nulls}")
    print()

# Business checks
df_c = spark.table("silver_countries")

print("Regions present:")
df_c.groupBy("region").count().orderBy("count", ascending=False).show()

print("Top 10 by population:")
df_c.select("country_name", "population", "region") \
    .orderBy("population", ascending=False).show(10)

StatementMeta(, 3f746ad1-7a07-4b94-af06-ccf436ba0d77, 11, Finished, Available, Finished, False)

=== SILVER LAYER VALIDATION ===

--- silver_countries ---
  Rows: 254
  Nulls in country_code_2: 0
  Nulls in region: 0
  Nulls in population: 0

--- silver_country_currencies ---
  Rows: 276
  Nulls in country_code_3: 1
  Nulls in currency_code: 0

--- silver_country_languages ---
  Rows: 427
  Nulls in country_code_3: 1
  Nulls in language_name: 0

Regions present:
+---------+-----+
|   region|count|
+---------+-----+
|   Africa|   60|
| Americas|   56|
|   Europe|   54|
|     Asia|   52|
|  Oceania|   27|
|Antarctic|    5|
+---------+-----+

Top 10 by population:
+-------------+----------+--------+
| country_name|population|  region|
+-------------+----------+--------+
|        India|1429404000|    Asia|
|        China|1404890000|    Asia|
|United States| 341784857|Americas|
|    Indonesia| 288315089|    Asia|
|     Pakistan| 241499431|    Asia|
|      Nigeria| 223800000|  Africa|
|       Brazil| 214211951|Americas|
|   Bangladesh| 169828911|    Asia|
|       Russia| 146028325|  Eur